In [60]:
from ase.io import read, write
from ase import Atom
from ase.neighborlist import neighbor_list
from soprano.collection import AtomsCollection
from soprano.collection.generate import defectGen, substitutionGen, additionGen
from soprano.properties.linkage import Bonds
from soprano.selection import AtomSelection
import numpy as np
import itertools

from scipy.special import comb
from weas_widget import WeasWidget

In [61]:
cha_base = read('CHA.cif')
SAR = 5  # target Si:Al ratio after substitution
si_sel = AtomSelection.from_element(cha_base, 'Si')
nSi_base = len(si_sel)
# nSi_final / nAl = SAR  =>  nAl = nSi_base / (SAR + 1)
n_Al = nSi_base // (SAR + 1)

print(f'Number of Si atoms: {nSi_base}')
print(f'Number of Al atoms to substitute: {n_Al}')
print(f'Actual SAR after substitution: {(nSi_base - n_Al) / n_Al:.2f}')
print(f'There are {comb(nSi_base, n_Al):.3e} possible structures')


def lowenstein_rule(atoms, indices):
    """Lowenstein's rule: Al-O-Al linkages are forbidden.
    Implemented as a minimum Al-Al distance threshold (< 4.0 Å rejected).
    """
    RCUT = 4.0  # Å
    al_indices = AtomSelection.from_element(atoms, 'Al').indices
    if len(al_indices) < 2:
        return True
    for i, idx in enumerate(al_indices):
        other = np.delete(al_indices, i)
        if atoms.get_distances(idx, other, mic=True).min() < RCUT:
            return False
    return True


max_attempts = 500
cha_with_al = AtomsCollection(substitutionGen(
    cha_base,
    'Al',
    to_replace=si_sel,
    n=n_Al,
    accept=lowenstein_rule,
    max_attempts=max_attempts,
    random=True,
))
print(f'{len(cha_with_al)} structures generated from a max of {max_attempts} attempts.')
print(f'(i.e. {max_attempts - len(cha_with_al)} were rejected by the acceptance rule)')

Number of Si atoms: 36
Number of Al atoms to substitute: 6
Actual SAR after substitution: 5.00
There are 1.948e+06 possible structures
62 structures generated from a max of 500 attempts.
(i.e. 438 were rejected by the acceptance rule)


In [62]:
WeasWidget(cha_with_al.structures)

WeasWidget(children=(<weas_widget.base_widget.BaseWidget object at 0x14e5e2810>,))

## A note on where the protons end up

`additionGen` places each H on a sphere of radius `add_r = 2.5 Å` around an Al,
and the acceptance rule only rejects contacts closer than 0.8 Å. **Nothing in
this step attaches the proton to an oxygen.** In the structures written below,
every H therefore sits ~2.3-2.5 Å from its nearest O, which is far too long to
be a chemical bond (a Brønsted O-H is ~0.98 Å).

This is deliberate: it is a cheap starting guess, not the final geometry. The
first geometry relaxation inside PyMC-NMR seats the protons properly. Relaxing
this structure with MACE moves all six H onto Al-O-Si bridges at O-H =
0.97-1.00 Å - genuine Brønsted acid sites - and lowers the energy by ~23 eV.

So do not be alarmed if the structures below look wrong in the viewer, or if
`basin.xyz` in Tutorial 1 appears to have free-floating hydrogens. They are
resolved by the first relaxation.


In [63]:
def h_accept_rule(atoms, indices):
    """Accept only if no H atom overlaps with any other atom (< 0.8 Å)."""
    h_indices = AtomSelection.from_element(atoms, 'H').indices
    for h_idx in h_indices:
        other = [i for i in range(len(atoms)) if i != h_idx]
        if atoms.get_distances(h_idx, other, mic=True).min() < 0.8:
            return False
    return True


def get_protonated_structures(
        atoms,
        add_r: float = 2.5,
        ndefects: int = 1,
        accept_rule=h_accept_rule,
        max_per_host: int = 3,
):
    """
    Add Brønsted H near each Al site using additionGen.

    Parameters
    ----------
    atoms : Atoms
        Al-substituted zeolite structure.
    add_r : float
        Radius from Al atoms at which to place H, in Angstroms.
    ndefects : int
        Number of H atoms to add (one per Al to charge-compensate).
    accept_rule : callable
        Acceptance function passed to additionGen.
    max_per_host : int
        Maximum number of structures to collect per host structure.
    """
    aG = additionGen(
        atoms,
        'H',
        to_addition=AtomSelection.from_element(atoms, 'Al'),
        add_r=add_r,
        n=ndefects,
        accept=accept_rule,
    )
    aColl = AtomsCollection(list(itertools.islice(aG, max_per_host)))
    ngenerated = len(aColl)
    if ngenerated == 0:
        print('No structures accepted for this host structure.')
        return AtomsCollection([])
    print(f'Generated {ngenerated: >5}; {len(aColl): >5} structures added to collection.')
    return aColl

In [64]:
structures_to_decorate = cha_with_al.structures
max_total = 20       # cap total protonated structures
max_per_host = 3     # cap per Al-substituted structure

cha_with_h = AtomsCollection()
for s in structures_to_decorate:
    if len(cha_with_h) >= max_total:
        break
    remaining = max_total - len(cha_with_h)
    coll_temp = get_protonated_structures(
        s,
        add_r=2.5,
        ndefects=n_Al,
        max_per_host=min(max_per_host, remaining),
    )
    cha_with_h += coll_temp

print(f'Accepted {len(cha_with_h)} protonated structures in total.')

Generated     3;     3 structures added to collection.
Generated     3;     3 structures added to collection.
Generated     3;     3 structures added to collection.
Generated     3;     3 structures added to collection.
Generated     3;     3 structures added to collection.
Generated     3;     3 structures added to collection.
Generated     2;     2 structures added to collection.
Accepted 20 protonated structures in total.


In [65]:
WeasWidget(cha_with_h.structures)

WeasWidget(children=(<weas_widget.base_widget.BaseWidget object at 0x119528530>,))

In [68]:
cha_with_h.structures[0].write("CHA_SAR5_with_H.cif")
cha_with_h.structures[0].write("CHA_SAR5_with_H.extxyz")


for i, s in enumerate(cha_with_h.structures):
    s.write(f"CHA_SAR5_with_H_{i:03d}.extxyz")
